# Spatial Analysis with Squidpy

This notebook performs comprehensive spatial analysis:
- Spatial neighborhood analysis
- Cell-cell interaction analysis  
- Spatial clustering
- Niche identification
- Spatial statistics

**Input:** Annotated AnnData from 02_phenotyping.ipynb

**Output:** Spatial metrics and interaction networks

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Rapids GPU-accelerated paths used in cells 10 (Moran's I) and 12 (niches).
import rapids_singlecell as rsc
import cupy as cp
from cuml.cluster import KMeans as cuKMeans
from cuml.neighbors import NearestNeighbors as cuNearestNeighbors

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f"Squidpy version: {sq.__version__}")
print(f"rapids_singlecell version: {rsc.__version__}")


In [ ]:
import os
import gc
from pathlib import Path

SAMPLE_ID   = os.environ.get("XENIUM_SAMPLE_ID", "0041323")
DATA_DIR    = Path(os.environ.get("XENIUM_OUTPUT_DIR",  f"../data/processed/{SAMPLE_ID}"))
OUTPUT_DIR  = DATA_DIR
FIGURES_DIR = Path(os.environ.get("XENIUM_FIGURES_DIR", f"../figures/{SAMPLE_ID}/03_spatial_analysis"))
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

input_file = DATA_DIR / f"{SAMPLE_ID}_phenotyped.h5ad"
if not input_file.exists():
    raise FileNotFoundError(
        f"Expected {input_file}. Run notebooks/02_phenotyping.ipynb first."
    )

print(f"Sample:    {SAMPLE_ID}")
print(f"Loading:   {input_file}")
adata = sc.read_h5ad(input_file)
print(f"Data shape after full load: {adata.shape}")

# Slim adata to reduce memory pressure for downstream squidpy ops
# Cgroup cap on this tunnel is 240 GB; full phenotyped adata + subsample copies OOMs.
_drop_obsm  = [k for k in ["X_pca", "X_scvi", "X_umap"] if k in adata.obsm]
_drop_obsp  = [k for k in ["connectivities", "distances", "scvi_connectivities",
                           "scvi_distances", "spatial_connectivities", "spatial_distances"]
               if k in adata.obsp]
_drop_layer = [k for k in ["log_normalized", "scaled"] if k in adata.layers]
_drop_uns   = [k for k in ["neighbors", "scvi", "pca", "umap",
                           "_scvi_manager_uuid", "_scvi_uuid",
                           "dendrogram_leiden_1.5", "leiden_1.5_nhood_enrichment",
                           "leiden_1.5_centrality_scores",
                           "rank_genes_leiden", "hvg", "log1p", "spatial_neighbors"]
               if k in adata.uns]
for k in _drop_obsm:  del adata.obsm[k]
for k in _drop_obsp:  del adata.obsp[k]
for k in _drop_layer: del adata.layers[k]
for k in _drop_uns:   del adata.uns[k]
gc.collect()

print(f"Dropped: obsm={_drop_obsm}, obsp={_drop_obsp}, layers={_drop_layer}, uns={_drop_uns}")
print(f"Kept obsm: {list(adata.obsm.keys())}")
print(f"Kept obsp: {list(adata.obsp.keys())}")
print(f"Kept layers: {list(adata.layers.keys())}")
print(f"Cell types: {adata.obs['celltype'].nunique()}")


## 1. Build Spatial Graph

In [ ]:
# Build spatial neighbor graph
sq.gr.spatial_neighbors(adata, coord_type='generic', n_neighs=10)
print("Spatial graph constructed")

# Visualize spatial connectivity (plain matplotlib — avoids sq.pl.spatial_scatter dependency on adata.uns['spatial'])
if 'spatial' in adata.obsm:
    import matplotlib as mpl
    xy = adata.obsm['spatial']
    # Subsample to 200k cells for plotting speed
    rng = np.random.default_rng(0)
    n = xy.shape[0]
    idx = rng.choice(n, size=min(200_000, n), replace=False)
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    cats = adata.obs['celltype'].astype('category').iloc[idx]
    codes = cats.cat.codes.to_numpy()
    cmap = mpl.colormaps.get_cmap('tab20').resampled(max(20, len(cats.cat.categories)))
    ax.scatter(xy[idx,0], xy[idx,1], c=codes, cmap=cmap, s=0.5, linewidths=0)
    ax.set_aspect('equal')
    ax.set_title(f'Spatial distribution of celltype ({len(idx):,}/{n:,} cells shown)')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'spatial_connectivity.png', dpi=300, bbox_inches='tight')
    plt.show()


## 2. Neighborhood Enrichment Analysis

In [ ]:
# nhood_enrichment uses sequential permutation (n_jobs=1) because n_jobs>1 hits a
# numba TypeError: joblib serializes inputs as readonly arrays which squidpy's
# numba kernel signature doesn't accept. n_perms reduced from default 1000 to 100
# to bound runtime to ~3-5 min (still gives p-value resolution to 0.01).
sq.gr.nhood_enrichment(adata, cluster_key="celltype", n_perms=100, n_jobs=1, seed=0)

# sq.pl.nhood_enrichment with method='ward' calls scipy.cluster.hierarchy.linkage on
# the z-score matrix. With rare cell types (e.g. Beta=602, B_cells=25) some pairs
# get NaN z-scores and linkage raises "condensed distance matrix must contain only
# finite values". Plot with sns.clustermap instead, NaN-filled with 0 — this gives
# the same dendrogram-clustered heatmap without the crash.
import seaborn as sns
import pandas as pd
nhood = adata.uns["celltype_nhood_enrichment"]
zscore = nhood["zscore"] if isinstance(nhood, dict) else nhood
cats = list(adata.obs["celltype"].cat.categories)
z_df = pd.DataFrame(zscore, index=cats, columns=cats)
n_nan = int(z_df.isna().sum().sum())
n_inf = int(np.isinf(z_df.values).sum())
if n_nan or n_inf:
    print(f"  cleaning {n_nan} NaN + {n_inf} inf z-scores -> 0 for plotting")
z_df = z_df.replace([np.inf, -np.inf], 0).fillna(0.0)
# Sanity assertion guards against scipy linkage's "non-finite" crash.
assert np.all(np.isfinite(z_df.values)), "z-score still has non-finite after cleanup"

try:
    g = sns.clustermap(
        z_df, method="ward", cmap="coolwarm",
        vmin=-50, vmax=50, center=0,
        figsize=(max(8, 0.55 * len(cats)), max(8, 0.55 * len(cats))),
        cbar_kws={"label": "neighborhood z-score"},
        dendrogram_ratio=0.12,
        annot=False,
    )
    g.ax_heatmap.set_xlabel("")
    g.ax_heatmap.set_ylabel("")
    g.fig.suptitle("Neighborhood enrichment z-score (Ward-clustered)", y=1.0)
except Exception as e:
    # Defensive fallback: linkage can still fail on degenerate matrices
    # (e.g. two identical rows -> zero distance -> NaN in cosine etc.).
    # Plain heatmap, alphabetic ordering, no dendrogram.
    print(f"  clustermap failed ({e!r}); falling back to plain heatmap")
    plt.close("all")
    fig, ax = plt.subplots(figsize=(max(8, 0.55*len(cats)), max(8, 0.55*len(cats))))
    sns.heatmap(z_df, cmap="coolwarm", vmin=-50, vmax=50, center=0,
                cbar_kws={"label": "neighborhood z-score"},
                annot=False, ax=ax)
    ax.set_title("Neighborhood enrichment z-score")
plt.savefig(FIGURES_DIR / "neighborhood_enrichment.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close("all")

print("Neighborhood enrichment calculated")


## 3. Co-occurrence Analysis

In [ ]:
# Calculate co-occurrence probability
N_SUB_CO = 150_000
if adata.n_obs > N_SUB_CO:
    rng = np.random.default_rng(0)
    sub_idx = rng.choice(adata.n_obs, size=N_SUB_CO, replace=False)
    adata_co = adata[sub_idx].copy()
    print(f"Co-occurrence computed on {N_SUB_CO:,}/{adata.n_obs:,} subsampled cells")
else:
    adata_co = adata

sq.gr.co_occurrence(adata_co, cluster_key='celltype')

# Custom renderer: clean 4-column grid with shared y-axis (no label overlap),
# simplified per-panel titles (just the cluster name instead of squidpy's
# default $\frac{p(exp|X)}{p(exp)}$ math), and a single shared legend on the
# right. Implemented inline so stage 03 stays self-contained.
def _plot_co_occurrence_clean(adata, cluster_key='celltype', out_path=None,
                              ncols=4, panel_size=(3.6, 2.8)):
    co = adata.uns[f'{cluster_key}_co_occurrence']
    out = co['occ']
    interval = co['interval'][1:]
    cats = list(adata.obs[cluster_key].cat.categories)
    n = len(cats)
    nrows = (n + ncols - 1) // ncols
    palette_key = f'{cluster_key}_colors'
    if palette_key in adata.uns and len(adata.uns[palette_key]) >= n:
        palette = list(adata.uns[palette_key])
    else:
        palette = list(plt.cm.tab20(np.linspace(0, 1, n)))
    color_map = {cat: palette[i] for i, cat in enumerate(cats)}
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(panel_size[0]*ncols, panel_size[1]*nrows),
                             sharey=True, sharex=True)
    axes = np.atleast_1d(axes).flatten()
    for j, target in enumerate(cats):
        ax = axes[j]
        for i, source in enumerate(cats):
            ax.plot(interval, out[j, i, :], color=color_map[source], lw=1.2)
        ax.set_title(target, fontsize=10)
        ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.6, alpha=0.6)
        ax.tick_params(labelsize=8)
        if j % ncols == 0:
            ax.set_ylabel('p(co-occur) / p(marginal)', fontsize=9)
        if j >= n - ncols:
            ax.set_xlabel('distance (μm)', fontsize=9)
    for j in range(n, len(axes)):
        axes[j].axis('off')
    handles = [plt.Line2D([0], [0], color=color_map[c], lw=2, label=c) for c in cats]
    fig.legend(handles=handles, loc='center right',
               bbox_to_anchor=(1.0, 0.5), frameon=False,
               title=cluster_key, fontsize=9, title_fontsize=10)
    fig.suptitle(f'Co-occurrence ratio vs distance, by {cluster_key}',
                 fontsize=12, y=0.99)
    fig.tight_layout(rect=(0, 0, 0.88, 0.97))
    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
    return fig

_plot_co_occurrence_clean(adata_co, cluster_key='celltype',
                          out_path=FIGURES_DIR / 'co_occurrence.png')
plt.show()
plt.close('all')

if 'celltype_co_occurrence' in adata_co.uns:
    adata.uns['celltype_co_occurrence'] = adata_co.uns['celltype_co_occurrence']
del adata_co
gc.collect()


## 4. Spatial Autocorrelation

In [ ]:
# Build log-normalized layer in place of the (dropped) scaled layer
if 'log_normalized' not in adata.layers:
    adata.X = adata.layers['counts'].copy()
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    adata.layers['log_normalized'] = adata.X.copy()

# rank_genes was already computed in 02_phenotyping.ipynb (saved in uns['rank_genes_leiden']).
# Skip here — only compute if missing.

# Moran's I spatial autocorrelation on top 100 genes by mean expression
# Subsample to 200k cells — Moran's I via permutation on 1.2M cells is expensive.
N_SUB_MI = 200_000
if adata.n_obs > N_SUB_MI:
    rng = np.random.default_rng(3)
    sub_idx = rng.choice(adata.n_obs, size=N_SUB_MI, replace=False)
    adata_mi = adata[sub_idx].copy()
    print(f"Morans I on {N_SUB_MI:,}/{adata.n_obs:,} cells")
else:
    adata_mi = adata

# pick 100 genes w/ highest mean counts
gene_means = adata_mi.layers['counts'].mean(axis=0)
try:
    gene_means = gene_means.A1
except AttributeError:
    pass
import numpy as _np
top_idx = _np.argsort(-_np.asarray(gene_means).ravel())[:100]
genes_to_test = list(adata_mi.var_names[top_idx])

# Need spatial_neighbors on the subsample
sq.gr.spatial_neighbors(adata_mi, coord_type='generic', n_neighs=10)
rsc.gr.spatial_autocorr(adata_mi, mode='moran', genes=genes_to_test, layer='log_normalized')  # GPU

# Copy Moran's I results back to main adata
if 'moranI' in adata_mi.uns:
    adata.uns['moranI'] = adata_mi.uns['moranI']

top_spatially_variable = adata.uns['moranI'].head(20)
print("\nTop 20 spatially variable genes:")
print(top_spatially_variable[['I', 'pval_norm']])

# Free the subsample
del adata_mi
gc.collect()


## 5. Spatial Clustering and Domains

In [ ]:
# Spatial niches via composition clustering + spatial smoothing (Banksy-style).
# 1. Each cell's vector = fraction of each celltype among its K=50 spatial neighbors.
# 2. cuML KMeans (GPU, n_clusters=12, n_init=10) on composition vectors.
# 3. Spatial smoothing: replace each cell's niche label with mode of neighbors' labels
#    (2 passes, k=30). This removes salt-and-pepper and yields contiguous tissue regions.
# cuML KMeans + NN replace sklearn (cell 1 imports).
import matplotlib as mpl

if 'spatial' in adata.obsm:
    coords = np.asarray(adata.obsm['spatial'], dtype=np.float64)
    finite = np.isfinite(coords).all(axis=1)
    if not finite.all():
        adata._inplace_subset_obs(finite)
        coords = np.asarray(adata.obsm['spatial'], dtype=np.float64)
    celltypes = adata.obs['celltype'].astype('category')
    ct_cats = list(celltypes.cat.categories)
    ct_codes = celltypes.cat.codes.to_numpy()
    n_cells = coords.shape[0]

    K_COMP = 50
    print(f"Building spatial k-NN (k={K_COMP}) over {n_cells:,} cells (cuML)...")
    nn = cuNearestNeighbors(n_neighbors=K_COMP, algorithm='brute', output_type='numpy')
    coords_f32 = coords.astype(np.float32)
    nn.fit(coords_f32)
    nbrs = nn.kneighbors(coords_f32, return_distance=False)
    neighbor_codes = ct_codes[nbrs]

    comp = np.zeros((n_cells, len(ct_cats)), dtype=np.float32)
    for code in range(len(ct_cats)):
        comp[:, code] = (neighbor_codes == code).sum(axis=1)
    comp /= K_COMP

    n_niches = 12
    print(f"cuML KMeans({n_niches}, n_init=10) on composition vectors (GPU)...")
    km = cuKMeans(n_clusters=n_niches, random_state=42,
                  max_iter=300, n_init=10, init='scalable-k-means++',
                  output_type='numpy')
    raw = km.fit_predict(comp.astype(np.float32)).astype(np.int32)
    print(f"raw niche sizes: {np.bincount(raw).tolist()}")

    K_SMOOTH = 30
    sm_nbrs = nbrs[:, :K_SMOOTH] if K_SMOOTH <= K_COMP else None
    if sm_nbrs is None:
        nns = cuNearestNeighbors(n_neighbors=K_SMOOTH, algorithm='brute', output_type='numpy')
        nns.fit(coords_f32)
        sm_nbrs = nns.kneighbors(coords_f32, return_distance=False)

    def majority_vote(labels, nb_idx):
        n_labels = int(labels.max()) + 1
        out = labels.copy()
        chunk = 50_000
        for i in range(0, len(labels), chunk):
            sl = slice(i, min(i + chunk, len(labels)))
            nbr_labels = labels[nb_idx[sl]]
            oh = np.eye(n_labels, dtype=np.int32)[nbr_labels]
            counts = oh.sum(axis=1)
            out[sl] = counts.argmax(axis=1)
        return out

    smoothed = raw
    for p in range(2):
        new = majority_vote(smoothed, sm_nbrs)
        print(f"  smoothing pass {p+1}: {(new != smoothed).sum():,} cells changed label")
        smoothed = new

    adata.obs['spatial_niche'] = pd.Categorical(smoothed.astype(str))
    print(f"final niche sizes:\n{adata.obs['spatial_niche'].value_counts().sort_index()}")

    rng = np.random.default_rng(0)
    idx = rng.choice(n_cells, size=min(300_000, n_cells), replace=False)

    try:
        from matplotlib.colors import ListedColormap
        from matplotlib.lines import Line2D

        # Discrete colormap so each integer niche label maps to a fixed color
        # (and the legend swatches match the scatter exactly).
        tab20 = mpl.colormaps.get_cmap('tab20')
        niche_colors = [tab20(i % 20) for i in range(n_niches)]
        niche_cmap = ListedColormap(niche_colors)

        # 4-column layout: niche map | niche legend | celltype map | celltype legend
        fig, axes = plt.subplots(
            1, 4, figsize=(24, 8),
            gridspec_kw={'width_ratios': [1.0, 0.28, 1.0, 0.38], 'wspace': 0.04},
        )
        ax_niche, ax_leg, ax_ct, ax_ct_leg = axes
        ax_leg.axis('off')
        ax_ct_leg.axis('off')

        ax_niche.scatter(coords[idx, 0], coords[idx, 1],
                         c=smoothed[idx], cmap=niche_cmap,
                         vmin=-0.5, vmax=n_niches - 0.5,
                         s=0.5, linewidths=0)
        ax_niche.set_title(f'Spatial niches (smoothed, K_COMP={K_COMP}, n_niches={n_niches})')
        ax_niche.set_aspect('equal', adjustable='datalim')

        niche_sizes = np.bincount(smoothed, minlength=n_niches)
        niche_handles = [
            Line2D([0], [0], marker='o', linestyle='', markersize=9,
                   markerfacecolor=niche_colors[i], markeredgecolor='none',
                   label=f'Niche {i}  (n={niche_sizes[i]:,})')
            for i in range(n_niches)
        ]
        ax_leg.legend(handles=niche_handles, title='Spatial niche',
                      loc='center', frameon=False,
                      fontsize=9, title_fontsize=10,
                      handletextpad=0.6, labelspacing=1.1,
                      borderaxespad=0.0)

        # Cell-types panel: ListedColormap so the legend swatches match exactly.
        n_ct = len(ct_cats)
        ct_colors = [tab20(i % 20) for i in range(n_ct)]
        ct_cmap = ListedColormap(ct_colors)

        sub_codes = celltypes.iloc[idx].cat.codes.to_numpy()
        ax_ct.scatter(coords[idx, 0], coords[idx, 1],
                      c=sub_codes, cmap=ct_cmap,
                      vmin=-0.5, vmax=n_ct - 0.5,
                      s=0.5, linewidths=0)
        ax_ct.set_title('Cell types')
        ax_ct.set_aspect('equal', adjustable='datalim')
        ax_ct.tick_params(axis='y', left=False, labelleft=False)

        ct_sizes = np.bincount(ct_codes, minlength=n_ct)
        ct_handles = [
            Line2D([0], [0], marker='o', linestyle='', markersize=9,
                   markerfacecolor=ct_colors[i], markeredgecolor='none',
                   label=f'{ct_cats[i]}  (n={ct_sizes[i]:,})')
            for i in range(n_ct)
        ]
        ax_ct_leg.legend(handles=ct_handles, title='Cell type',
                         loc='center', frameon=False,
                         fontsize=9, title_fontsize=10,
                         handletextpad=0.6, labelspacing=1.1,
                         borderaxespad=0.0)

        fig.savefig(FIGURES_DIR / 'spatial_domains.png', dpi=300, bbox_inches='tight')
        plt.close(fig)
    except Exception as e:
        print(f'spatial_domains plot failed: {e!r}')

    try:
        # 2-panel niche × cell-type matrix
        # left  : niche composition (rows = niches sum to 1)
        # right : celltype distribution across niches (rows = celltypes sum to 1)
        niche_str = pd.Series(smoothed.astype(str), name='niche', index=celltypes.index)
        comp = pd.crosstab(niche_str, celltypes, normalize='index').fillna(0.0)
        comp = comp.reindex(sorted(comp.index, key=int))
        dist = pd.crosstab(niche_str, celltypes, normalize='columns').fillna(0.0).T
        dist = dist.reindex(columns=sorted(dist.columns, key=int))
        abund = celltypes.value_counts()
        dist = dist.reindex(index=[c for c in abund.index if c in dist.index])

        fig2, axes2 = plt.subplots(
            1, 2, figsize=(22, 8),
            gridspec_kw={'width_ratios': [1.05, 1.0], 'wspace': 0.25},
        )
        sns.heatmap(comp, cmap='viridis', ax=axes2[0],
                    annot=True, fmt='.2f', annot_kws={'size': 7},
                    cbar_kws={'label': 'fraction of niche', 'shrink': 0.7},
                    linewidths=0.4, linecolor='white')
        axes2[0].set_title('Niche composition  (rows sum = 1)', fontsize=12)
        axes2[0].set_xlabel('cell type')
        axes2[0].set_ylabel('spatial niche')
        axes2[0].tick_params(axis='x', labelrotation=45, labelsize=9)
        for label in axes2[0].get_xticklabels():
            label.set_horizontalalignment('right')
        axes2[0].tick_params(axis='y', labelrotation=0, labelsize=10)

        sns.heatmap(dist, cmap='magma', ax=axes2[1],
                    annot=True, fmt='.2f', annot_kws={'size': 7},
                    cbar_kws={'label': 'fraction of celltype', 'shrink': 0.7},
                    linewidths=0.4, linecolor='white')
        axes2[1].set_title('Cell-type distribution across niches  (rows sum = 1)',
                          fontsize=12)
        axes2[1].set_xlabel('spatial niche')
        axes2[1].set_ylabel('cell type')
        axes2[1].tick_params(axis='x', labelrotation=0, labelsize=10)
        axes2[1].tick_params(axis='y', labelrotation=0, labelsize=10)

        fig2.suptitle('Spatial niche × cell type composition',
                      fontsize=13, y=1.02)
        fig2.savefig(FIGURES_DIR / 'niche_celltype_matrix.png',
                     dpi=200, bbox_inches='tight')
        plt.close(fig2)
    except Exception as e:
        print(f'niche × celltype matrix plot failed: {e!r}')


## 6. Ligand-Receptor Interaction Analysis

In [ ]:
# Ligand-receptor interaction analysis at the LINEAGE level (general celltype
# categories: Endocrine, Exocrine, Immune, Stromal, Vascular, Neural, Cycling)
# instead of subtype-level. Subtype-level was unbalanced for this panel —
# Beta/Gamma/Epsilon are too rare (n<50) for stable per-cluster mean expression
# and meaningful permutation null, so only Alpha + Delta survived a sane
# cell-count threshold. Aggregating to lineages eliminates the imbalance and
# shrinks the cluster-pair space ~15× (49 vs 729).
#
# Output: one dotplot PER source lineage (7 PNGs), each <200 KB. Mirrored in
# scripts/regen_ligrec_lineage.py for ad-hoc regeneration without re-running
# the notebook.
LINEAGE_MAP = {
    'Alpha': 'Endocrine', 'Beta': 'Endocrine', 'Delta': 'Endocrine',
    'Gamma': 'Endocrine', 'Epsilon': 'Endocrine',
    'Endocrine': 'Endocrine', 'Endocrine_pan': 'Endocrine',
    'Acinar': 'Exocrine', 'Ductal': 'Exocrine',
    'T_cells': 'Immune', 'B_cells': 'Immune', 'Myeloid': 'Immune',
    'Stellate': 'Stromal',
    'Endothelial': 'Vascular',
    'Schwann': 'Neural',
    'Proliferating': 'Cycling',
}

# Build LR subsample with celltype_general and Indeterminate* dropped.
ct_str = adata.obs['celltype'].astype(str)
keep_mask = ~ct_str.str.startswith('Indeterminate') & ct_str.isin(LINEAGE_MAP.keys())
print(f"  ligrec: dropping {int((~keep_mask).sum()):,} Indeterminate*/unmapped, "
      f"keeping {int(keep_mask.sum()):,}")

N_SUB_LR = 100_000
keep_idx = np.where(keep_mask.to_numpy())[0]
if keep_idx.size > N_SUB_LR:
    rng = np.random.default_rng(1)
    sub_idx = rng.choice(keep_idx, size=N_SUB_LR, replace=False)
else:
    sub_idx = keep_idx
adata_lr = adata[sub_idx].copy()
adata_lr.obs['celltype_general'] = pd.Categorical(
    adata_lr.obs['celltype'].astype(str).map(LINEAGE_MAP),
    categories=sorted(set(LINEAGE_MAP.values())),
)
print(f"  ligrec on {adata_lr.n_obs:,} cells; lineages = "
      f"{list(adata_lr.obs['celltype_general'].cat.categories)}")
print('  per-lineage cell counts:')
for lin, n in adata_lr.obs['celltype_general'].value_counts().items():
    print(f"      {lin:<10} {int(n):>7,}")

sq.gr.ligrec(
    adata_lr,
    cluster_key='celltype_general',
    n_perms=50,
    copy=False,
    use_raw=False,
)
print("Ligand-receptor analysis completed (lineage-level)")

if 'celltype_general_ligrec' in adata_lr.uns:
    adata.uns['celltype_general_ligrec'] = adata_lr.uns['celltype_general_ligrec']
    print(f"  ligrec pvalues shape: {adata_lr.uns['celltype_general_ligrec']['pvalues'].shape}")

# Per-source dotplots — 7 small PNGs.
try:
    TOP_N_PER_SOURCE = 40
    MIN_MEAN = 0.3
    PVAL_THRESH = 0.001
    ALPHA = 1e-4
    P_FLOOR = 1e-12
    LR_DPI = 130

    lr = adata.uns['celltype_general_ligrec']
    means = lr['means']
    pvals = lr['pvalues']
    metadata = lr.get('metadata')

    pvals_clean = np.where(np.isfinite(pvals.to_numpy(dtype=np.float64)),
                            np.maximum(pvals.to_numpy(dtype=np.float64), P_FLOOR),
                            1.0)
    means_arr = np.where(np.isfinite(means.to_numpy(dtype=np.float64)),
                          means.to_numpy(dtype=np.float64), 0.0)

    lineages = list(adata_lr.obs['celltype_general'].cat.categories)
    for src in lineages:
        col_mask = np.array([c1 == src for c1, c2 in means.columns])
        if not col_mask.any():
            continue
        m_src = means.loc[:, col_mask]
        p_src = pvals.loc[:, col_mask]
        means_src = means_arr[:, col_mask]
        pvals_src = pvals_clean[:, col_mask]

        sig_any = ((pvals_src < PVAL_THRESH) & (means_src >= MIN_MEAN)).any(axis=1)
        if sig_any.sum() == 0:
            print(f"  {src}: no significant L-R pairs — skipping")
            continue
        m_src = m_src.loc[sig_any]
        p_src = p_src.loc[sig_any]
        score = (-np.log10(pvals_src[sig_any])).max(axis=1)
        order = np.argsort(-score)[:TOP_N_PER_SOURCE]
        m_top = m_src.iloc[order].copy()
        p_top = p_src.iloc[order].copy()

        res = {'means': m_top, 'pvalues': p_top}
        if metadata is not None:
            res['metadata'] = metadata.reindex(m_top.index).copy()

        n_lr = m_top.shape[0]
        n_cp = m_top.shape[1]
        fig_w = max(6.0, 0.55 * n_cp + 4.0)
        fig_h = max(5.0, 0.18 * n_lr + 2.0)

        sq.pl.ligrec(
            res,
            means_range=(MIN_MEAN, np.inf),
            pvalue_threshold=PVAL_THRESH,
            alpha=ALPHA,
            remove_empty_interactions=True,
            swap_axes=True,
            title=f'{SAMPLE_ID} — {src} → all targets  (top {n_lr} L-R interactions)',
            figsize=(fig_w, fig_h),
            dpi=LR_DPI,
        )
        out = FIGURES_DIR / f'ligand_receptor_dotplot_{src}.png'
        plt.savefig(out, dpi=LR_DPI, bbox_inches='tight')
        plt.close('all')
        print(f"  {src}: wrote {out.name}  ({n_lr} pairs × {n_cp} targets)")

    # Remove legacy monolithic file if it exists from a previous run.
    legacy = FIGURES_DIR / 'ligand_receptor_dotplot.png'
    if legacy.exists():
        legacy.unlink()
        print(f"  removed legacy {legacy.name}")
except Exception as e:
    print(f'ligrec dotplots failed: {e!r}')

del adata_lr
gc.collect()

## 7. Ripley's Statistics

In [ ]:
# Ripley's L-function — subsampled for tractability
N_SUB_RI = 100_000
if adata.n_obs > N_SUB_RI:
    rng = np.random.default_rng(2)
    sub_idx = rng.choice(adata.n_obs, size=N_SUB_RI, replace=False)
    adata_ri = adata[sub_idx].copy()
    print(f"Ripley on {N_SUB_RI:,}/{adata.n_obs:,} cells")
else:
    adata_ri = adata

mode = 'L'
sq.gr.ripley(adata_ri, cluster_key='celltype', mode=mode, max_dist=500)

# sq.pl.ripley in this squidpy version does not accept a `clusters=` filter.
# Plot all clusters; pass figure size via figsize.
sq.pl.ripley(adata_ri, cluster_key='celltype', mode=mode, figsize=(12, 6))
plt.savefig(FIGURES_DIR / 'ripley_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

if 'ripley_celltype' in adata_ri.uns:
    adata.uns['ripley_celltype'] = adata_ri.uns['ripley_celltype']

del adata_ri
gc.collect()


## 8. Save Results

In [ ]:
# Save results
# sq.gr.ligrec stores DataFrames with MultiIndex columns; h5py requires str keys,
# so export those as CSVs and strip them from uns before write_h5ad.

# 1) Export ligrec MultiIndex DataFrames to CSV, then remove from uns.
# Filename convention: {SAMPLE_ID}_ligrec_lineage_{means,pvalues,metadata}.csv
# (lineage-level — celltype_general key, see cell 14).
lr_key = 'celltype_general_ligrec'
if lr_key in adata.uns and isinstance(adata.uns[lr_key], dict):
    lr = adata.uns[lr_key]
    for sub_key in list(lr.keys()):
        obj = lr[sub_key]
        if hasattr(obj, 'to_csv'):
            csv_path = OUTPUT_DIR / f"{SAMPLE_ID}_ligrec_lineage_{sub_key}.csv"
            try:
                obj.to_csv(csv_path)
                print(f"ligrec/{sub_key} -> {csv_path.name}")
            except Exception as e:
                print(f"ligrec/{sub_key} CSV failed: {e!r}")
    del adata.uns[lr_key]

# 2) Similar defensive cleanup for ripley
if 'ripley_celltype' in adata.uns and isinstance(adata.uns['ripley_celltype'], dict):
    rip = adata.uns['ripley_celltype']
    for sub_key in list(rip.keys()):
        obj = rip[sub_key]
        if hasattr(obj, 'to_csv'):
            csv_path = OUTPUT_DIR / f"{SAMPLE_ID}_ripley_{sub_key}.csv"
            try:
                obj.to_csv(csv_path)
                print(f"ripley/{sub_key} -> {csv_path.name}")
            except Exception as e:
                print(f"ripley/{sub_key} CSV failed: {e!r}")

# 3) co_occurrence: array, should be fine but handle numpy edge
co_key = 'co_occurrence'
if co_key in adata.uns and isinstance(adata.uns[co_key], dict):
    co = adata.uns[co_key]
    if 'occ' in co and hasattr(co['occ'], 'shape'):
        np.save(OUTPUT_DIR / f"{SAMPLE_ID}_co_occurrence.npy", co['occ'])
        print(f"co_occurrence array shape {co['occ'].shape} -> .npy")

output_file = OUTPUT_DIR / f"{SAMPLE_ID}_spatial_analysis.h5ad"
try:
    adata.write_h5ad(output_file)
    print(f"\nSpatial analysis results saved to: {output_file}")
except Exception as e:
    # Last-resort fallback: drop any remaining non-serializable uns entries
    print(f"write_h5ad failed ({e!r}); stripping uns entries and retrying")
    bad = []
    for k, v in list(adata.uns.items()):
        try:
            import h5py, tempfile
            with tempfile.NamedTemporaryFile(suffix='.h5', delete=True) as tmp:
                with h5py.File(tmp.name, 'w') as f:
                    pass
            _ = str(v)  # cheap serializability hint
        except Exception:
            bad.append(k)
    # Brute force: only keep simple types in uns
    safe = {}
    for k, v in adata.uns.items():
        if isinstance(v, (str, int, float, bool, list, tuple, dict, np.ndarray)):
            safe[k] = v
    adata.uns = safe
    adata.write_h5ad(output_file)
    print(f"Saved after sanitizing uns: {output_file}")

try:
    if 'celltype_nhood_enrichment' in adata.uns:
        nhood = adata.uns['celltype_nhood_enrichment']
        # squidpy may store either dict-with-'names' or just zscore array. Be defensive.
        zscore = nhood.get('zscore') if isinstance(nhood, dict) else nhood
        if zscore is not None:
            cats = list(adata.obs['celltype'].cat.categories) if hasattr(adata.obs['celltype'], 'cat') else None
            try:
                nhood_df = pd.DataFrame(zscore, index=cats, columns=cats)
            except Exception:
                nhood_df = pd.DataFrame(zscore)
            nhood_df.to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_neighborhood_enrichment.csv")
            print("Neighborhood enrichment exported")
except Exception as e:
    print(f"nhood CSV export failed: {e!r}")

try:
    if 'moranI' in adata.uns:
        adata.uns['moranI'].to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_morans_i.csv")
        print("Moran's I statistics exported")
except Exception as e:
    print(f"moranI CSV export failed: {e!r}")

# Export niche composition too
try:
    if 'spatial_niche' in adata.obs.columns:
        niche_df = adata.obs[['celltype', 'spatial_niche']]
        niche_df.to_csv(OUTPUT_DIR / f"{SAMPLE_ID}_spatial_niches.csv")
        print("Spatial niche assignments exported")
except Exception as e:
    print(f"niche CSV export failed: {e!r}")